# Bible Arc Visualization — Etsy Light Print Edition

Light-background companion to `02_arc_visualization.ipynb`, tuned for the Etsy shop.

Outputs three variants of the cross-testament arc graphic:

1. **Cream** (`#f8f5ee`) — warm, book-like default
2. **Pure white** (`#ffffff`) — crisp modern look
3. **Transparent** — for customers who want to composite onto their own background

Arc colors are re-tuned from the dark version: deep indigo for OT references, oxblood/crimson for NT. These sit well on light paper stocks and don't wash out the way the dark-bg blue/tan do.

Rendering uses the same Haydock cross-testament filter as the existing listing, so the topology of the print matches the dark version 1:1 — only the palette changes.

In [1]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from src.db.connection import get_connection
from src.data.book_mapping import CPDV_BOOK_ID_TO_ABBREV

In [2]:
conn = get_connection()
conn.connect()
conn.verify()
print('Connected to Neo4j')

Connected to Neo4j


## Book order and verse positions

Same setup as the dark notebook — CPDV 73-book canon, OT = books 1–46, NT = 47–73.

In [4]:
BOOK_ORDER = [CPDV_BOOK_ID_TO_ABBREV[i] for i in range(1, 74)]
OT_BOOKS = set(BOOK_ORDER[:46])
NT_BOOKS = set(BOOK_ORDER[46:])

with conn.session() as session:
    result = session.run('MATCH (v:Verse) RETURN v.book_id AS book, count(v) AS c ORDER BY book')
    verse_counts = {r['book']: r['c'] for r in result}

book_start_positions = {}
pos = 0
for book in BOOK_ORDER:
    book_start_positions[book] = pos
    pos += verse_counts.get(book, 0)
total_verses = pos
print(f'Total verses: {total_verses:,}')

Total verses: 35,817


In [5]:
print('Building verse position index...')
with conn.session() as session:
    result = session.run('''
        MATCH (v:Verse)
        RETURN v.id AS verse_id, v.book_id AS book
        ORDER BY v.book_id, v.chapter, v.verse
    ''')
    verse_positions = {}
    counter = {b: 0 for b in BOOK_ORDER}
    for r in result:
        b = r['book']
        if b in book_start_positions:
            verse_positions[r['verse_id']] = book_start_positions[b] + counter[b]
            counter[b] += 1
print(f'Indexed {len(verse_positions):,} verses')

Building verse position index...
Indexed 35,817 verses


## Query cross-references

Locked to **Haydock + cross-testament only** to match the current Etsy listing. Edit the filters if you want a different print.

In [6]:
SOURCE_FILTER = ['Haydock']
TESTAMENT_FILTER = 'cross_only'  # 'all', 'cross_only', 'ot_to_nt', 'nt_to_ot', 'same_only'

query = '''
    MATCH (a:Verse)-[r:CROSS_REFERENCES]->(b:Verse)
    WHERE any(s IN r.sources WHERE s IN $sources)
    RETURN a.id AS from_id, a.book_id AS from_book,
           b.id AS to_id, b.book_id AS to_book
'''

with conn.session() as session:
    result = session.run(query, sources=SOURCE_FILTER)
    crossrefs = []
    for r in result:
        fid, tid = r['from_id'], r['to_id']
        if fid not in verse_positions or tid not in verse_positions:
            continue
        fb, tb = r['from_book'], r['to_book']
        f_ot, t_ot = fb in OT_BOOKS, tb in OT_BOOKS
        if TESTAMENT_FILTER == 'cross_only' and f_ot == t_ot:
            continue
        if TESTAMENT_FILTER == 'ot_to_nt' and not (f_ot and not t_ot):
            continue
        if TESTAMENT_FILTER == 'nt_to_ot' and not (not f_ot and t_ot):
            continue
        if TESTAMENT_FILTER == 'same_only' and f_ot != t_ot:
            continue
        crossrefs.append({
            'from_pos': verse_positions[fid],
            'to_pos': verse_positions[tid],
            'to_book': tb,
        })

from_positions = np.array([c['from_pos'] for c in crossrefs])
to_positions = np.array([c['to_pos'] for c in crossrefs])
is_to_nt = np.array([c['to_book'] in NT_BOOKS for c in crossrefs])
print(f'Loaded {len(crossrefs):,} cross-references')

Loaded 1,534 cross-references


## Light-palette arc renderer

Key differences from the dark-bg version:

- **Colors**: `#1e2a5e` deep indigo (OT) and `#7a1f2b` oxblood (NT). Both are dark enough to register on cream/white at low alpha without looking muddy.
- **Higher alpha**: light backgrounds need ~2–3× more opacity to match the visual density of the dark print, because dark ink on light is a subtractive effect while bright lines on dark are additive.
- **Baseline + ticks** drawn in warm charcoal (`#2a2a2a`) instead of white.

In [7]:
def create_arc_points(x1, x2, num_points=100):
    center = (x1 + x2) / 2
    radius = abs(x2 - x1) / 2
    if radius == 0:
        return None, None
    if x2 > x1:
        theta = np.linspace(np.pi, 0, num_points)
        y_sign = 1
    else:
        theta = np.linspace(0, np.pi, num_points)
        y_sign = -1
    x = center + radius * np.cos(theta)
    y = y_sign * radius * np.sin(theta)
    return x, y

In [8]:
# Light palette presets
PALETTES = {
    'cream': {
        'bg': '#f8f5ee',
        'ot': '#1e2a5e',  # deep indigo
        'nt': '#7a1f2b',  # oxblood
        'ink': '#2a2a2a',
    },
    'white': {
        'bg': '#ffffff',
        'ot': '#14265c',
        'nt': '#6e1520',
        'ink': '#222222',
    },
    'transparent': {
        'bg': None,      # figure bg will be transparent on save
        'ot': '#1e2a5e',
        'nt': '#7a1f2b',
        'ink': '#2a2a2a',
    },
}

In [9]:
def draw_bible_arcs_light(from_pos, to_pos, is_to_nt,
                          palette='cream',
                          figsize=(40, 16),
                          alpha=0.08,
                          linewidth=0.4,
                          sample_size=None,
                          show_book_labels=True,
                          arc_smoothness=100,
                          margin_percent=2,
                          respect_figsize=False,
                          arc_height_scale=1.0):
    """Light-background print renderer. See notebook header for palette notes."""
    p = PALETTES[palette]
    bg = p['bg'] if p['bg'] is not None else 'none'

    fig, ax = plt.subplots(figsize=figsize)
    if p['bg'] is not None:
        fig.patch.set_facecolor(p['bg'])
        ax.set_facecolor(p['bg'])
    else:
        fig.patch.set_alpha(0)
        ax.set_facecolor('none')

    if sample_size and sample_size < len(from_pos):
        idx = np.random.choice(len(from_pos), sample_size, replace=False)
        from_pos = from_pos[idx]
        to_pos = to_pos[idx]
        is_to_nt = is_to_nt[idx]

    n = len(from_pos)
    print(f'Drawing {n:,} arcs on {palette} background...')

    ot_segments, nt_segments = [], []
    for i in range(n):
        x, y = create_arc_points(from_pos[i], to_pos[i], arc_smoothness)
        if x is None:
            continue
        if respect_figsize:
            y = y * arc_height_scale
        points = np.column_stack([x, y])
        segs = np.array([points[:-1], points[1:]]).transpose(1, 0, 2)
        (nt_segments if is_to_nt[i] else ot_segments).extend(segs)

    if ot_segments:
        ax.add_collection(LineCollection(ot_segments, colors=p['ot'], alpha=alpha, linewidths=linewidth))
    if nt_segments:
        ax.add_collection(LineCollection(nt_segments, colors=p['nt'], alpha=alpha, linewidths=linewidth))

    ax.axhline(y=0, color=p['ink'], linewidth=1, alpha=0.65)

    if show_book_labels:
        for book in BOOK_ORDER:
            pos = book_start_positions[book]
            ax.axvline(x=pos, color=p['ink'], linewidth=0.3, alpha=0.7, ymin=0.47, ymax=0.51)
        major_books = [
            'GEN','EXO','LEV','NUM','DEU','JOS','JDG','1SA','2SA','1KI','2KI',
            '1CH','2CH','EZR','NEH','TOB','EST','1MA','2MA','JOB','PSA','PRO',
            'ECC','WIS','SIR','ISA','JER','BAR','EZK','DAN','HOS','ZEC',
            'MAT','MRK','LUK','JHN','ACT','ROM','1CO','EPH','HEB','REV',
        ]
        for book in major_books:
            if book in book_start_positions:
                x = book_start_positions[book] + verse_counts.get(book, 0) / 2
                ax.text(x, -total_verses * 0.02, book,
                        ha='center', va='top', fontsize=8,
                        color=p['ink'], alpha=0.8, rotation=45)

    nt_start = book_start_positions['MAT']
    ax.axvline(x=nt_start, color=p['ink'], linewidth=2, alpha=0.35, linestyle='--')

    margin = total_verses * (margin_percent / 100)
    max_arc = total_verses / 2
    ax.set_xlim(-margin, total_verses + margin)
    ax.set_ylim(-max_arc * 0.6, max_arc * 0.6)
    ax.set_aspect('auto' if respect_figsize else 'equal')
    ax.axis('off')
    plt.tight_layout()
    return fig, ax

## Quick preview (sample)

10k-arc sample to verify colors and density before committing to a full 300 DPI render.

In [ ]:
fig, ax = draw_bible_arcs_light(from_positions, to_positions, is_to_nt,
                                palette='cream',
                                figsize=(30, 12),
                                sample_size=min(10000, len(from_positions)),
                                alpha=0.2,
                                linewidth=0.6)
plt.savefig('bible_arcs_light_preview_cream.png', dpi=150,
            facecolor=PALETTES['cream']['bg'], bbox_inches='tight')
plt.show()

## Full-density renders

Alpha is auto-tuned to the reference count, then scaled ~2.5× relative to the dark version because dark-ink-on-light needs more opacity to match perceptual density.

In [ ]:
auto_alpha = max(0.02, min(0.2, 3000 / len(crossrefs)))
light_alpha = min(0.5, auto_alpha * 2.5)
print(f'dark-version alpha: {auto_alpha:.3f}  ->  light alpha: {light_alpha:.3f}')

In [ ]:
# Cream — primary Etsy variant
fig, ax = draw_bible_arcs_light(from_positions, to_positions, is_to_nt,
                                palette='cream',
                                figsize=(40, 16),
                                alpha=light_alpha,
                                linewidth=0.4)
plt.savefig('bible_arcs_light_full_cream.png', dpi=200,
            facecolor=PALETTES['cream']['bg'], bbox_inches='tight')
plt.show()

In [ ]:
# Pure white variant
fig, ax = draw_bible_arcs_light(from_positions, to_positions, is_to_nt,
                                palette='white',
                                figsize=(40, 16),
                                alpha=light_alpha,
                                linewidth=0.4)
plt.savefig('bible_arcs_light_full_white.png', dpi=200,
            facecolor=PALETTES['white']['bg'], bbox_inches='tight')
plt.show()

In [ ]:
# Transparent variant — no facecolor, transparent=True on save
fig, ax = draw_bible_arcs_light(from_positions, to_positions, is_to_nt,
                                palette='transparent',
                                figsize=(40, 16),
                                alpha=light_alpha,
                                linewidth=0.4)
plt.savefig('bible_arcs_light_full_transparent.png', dpi=200,
            transparent=True, bbox_inches='tight')
plt.show()

## Print-size exports (300 DPI)

Matches the sizes currently listed in the dark version. Warning: each file is large and slow to render.

In [15]:
PRINT_SIZES = [
    (36, 24),
    (36, 30),
    (36, 36),
    (30, 24),
    (24, 18),
]
PRINT_DPI = 300
ARC_HEIGHT_SCALE = 1.0  # bump > 1 for taller canvases

for palette in ['cream', 'white', 'transparent']:
    p = PALETTES[palette]
    for w, h in PRINT_SIZES:
        print(f'Rendering {palette} {w}x{h}...')
        fig, ax = draw_bible_arcs_light(from_positions, to_positions, is_to_nt,
                                        palette=palette,
                                        figsize=(w, h),
                                        alpha=light_alpha,
                                        linewidth=0.4,
                                        respect_figsize=True,
                                        arc_height_scale=ARC_HEIGHT_SCALE)
        fname = f'bible_arcs_print_{palette}_{w}x{h}.png'
        if palette == 'transparent':
            plt.savefig(fname, dpi=PRINT_DPI, transparent=True, bbox_inches='tight')
        else:
            plt.savefig(fname, dpi=PRINT_DPI, facecolor=p['bg'], bbox_inches='tight')
        plt.close(fig)
        print(f'  saved {fname}')

Rendering cream 36x24...
Drawing 1,534 arcs on cream background...
  saved bible_arcs_print_cream_36x24.png
Rendering cream 36x30...
Drawing 1,534 arcs on cream background...
  saved bible_arcs_print_cream_36x30.png
Rendering cream 36x36...
Drawing 1,534 arcs on cream background...
  saved bible_arcs_print_cream_36x36.png
Rendering cream 30x24...
Drawing 1,534 arcs on cream background...
  saved bible_arcs_print_cream_30x24.png
Rendering cream 24x18...
Drawing 1,534 arcs on cream background...
  saved bible_arcs_print_cream_24x18.png
Rendering white 36x24...
Drawing 1,534 arcs on white background...
  saved bible_arcs_print_white_36x24.png
Rendering white 36x30...
Drawing 1,534 arcs on white background...
  saved bible_arcs_print_white_36x30.png
Rendering white 36x36...
Drawing 1,534 arcs on white background...
  saved bible_arcs_print_white_36x36.png
Rendering white 30x24...
Drawing 1,534 arcs on white background...
  saved bible_arcs_print_white_30x24.png
Rendering white 24x18...
Dra

## Cleanup

In [16]:
conn.close()
print('Connection closed')

Connection closed
